# Агуулахын хүргэлтийн робот — туршилтын notebook (X–XIII долоо хоног)
Үндсэн код `warehouse/` модульд, энд зөвхөн дуудаж, үр дүнг харна.

In [1]:
import pandas as pd, random
from warehouse.maps import load, load_all, MAIN_MAPS
from warehouse.astar import astar, bfs
from warehouse.vi import value_iteration, run_episode
pd.set_option('display.width', 200)

## X долоо хоног — орчин ба 9 зураглал

In [2]:
for e in load_all():
    print(e.name, f'{e.width}x{e.height}', 'чөлөөт нүд:', len(e.states()))
print()
print(load('m05_a').render())

m05_a 5x5 чөлөөт нүд: 20
m05_b 5x5 чөлөөт нүд: 20
m05_c 5x5 чөлөөт нүд: 18
m10_a 10x10 чөлөөт нүд: 58
m10_b 10x10 чөлөөт нүд: 73
m10_c 10x10 чөлөөт нүд: 72
m20_a 20x20 чөлөөт нүд: 196
m20_b 20x20 чөлөөт нүд: 286
m20_c 20x20 чөлөөт нүд: 297

S....
.##..
.##..
...#.
....G


In [3]:
e = load('m05_a')
print('(1,3)-аас U, p=0.8:', e.transitions((1, 3), 'U', 0.8))
print('(0,0)-оос U, p=0.8 (хил):', e.transitions((0, 0), 'U', 0.8))

(1,3)-аас U, p=0.8: [(0.8, (1, 3)), (0.09999999999999998, (0, 3)), (0.09999999999999998, (2, 3))]
(0,0)-оос U, p=0.8 (хил): [(0.9, (0, 0)), (0.09999999999999998, (1, 0))]


## XI долоо хоног — A* ба BFS тулгалт

In [4]:
rows = []
for e in load_all():
    a, b, f = astar(e), bfs(e), astar(e, tie_break='fifo')
    rows.append(dict(map=e.name, cost=a.cost, bfs_cost=b.cost, astar_exp=a.expanded, astar_fifo_exp=f.expanded, bfs_exp=b.expanded))
pd.DataFrame(rows)

,map,cost,bfs_cost,astar_exp,astar_fifo_exp,bfs_exp
0,m05_a,8,8,9,20,20
1,m05_b,12,12,16,20,20
2,m05_c,11,11,12,12,12
3,m10_a,18,18,19,58,58
4,m10_b,45,45,72,73,73
5,m10_c,18,18,56,61,70
6,m20_a,38,38,39,196,196
7,m20_b,133,133,228,248,249
8,m20_c,38,38,103,131,250


In [5]:
print(load('m10_b').render(path=astar(load('m10_b')).path))

S*#*****#G
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.*#*.#.*#*
.***.#.***


## XIII долоо хоног — Value Iteration
### Гар тооцоотой тулгах (1×3 коридор `S . G`)

In [6]:
from warehouse.env import GridWorld
c = GridWorld.from_string('S.G')
for p in (1.0, 0.8):
    for k in (1, 2, 3):
        r = value_iteration(c, p=p, max_iter=k)
        print(f'p={p} k={k}: V(S)={r.V[(0,0)]:.4f}  V(M)={r.V[(1,0)]:.4f}')
    r = value_iteration(c, p=p)
    print(f'p={p} нийлсэн ({r.iterations} давталт): V(S)={r.V[(0,0)]:.4f}  V(M)={r.V[(1,0)]:.4f}\n')

p=1.0 k=1: V(S)=-1.0000  V(M)=50.0000
p=1.0 k=2: V(S)=46.5000  V(M)=50.0000
p=1.0 k=3: V(S)=46.5000  V(M)=50.0000
p=1.0 нийлсэн (3 давталт): V(S)=46.5000  V(M)=50.0000

p=0.8 k=1: V(S)=-1.0000  V(M)=39.8000
p=0.8 k=2: V(S)=29.0580  V(M)=47.3620
p=0.8 k=3: V(S)=40.5161  V(M)=48.7988
p=0.8 нийлсэн (14 давталт): V(S)=44.8682  V(M)=49.1358



In [7]:
rows = []
for e in load_all():
    for p in (1.0, 0.8):
        r = value_iteration(e, p=p)
        rows.append(dict(map=e.name, p=p, iterations=r.iterations, converged=r.converged, V_start=round(r.V[e.start], 3), plan_ms=round(r.plan_ms, 1)))
pd.DataFrame(rows)

,map,p,iterations,converged,V_start,plan_ms
0,m05_a,1.0,9,True,28.884,0.9
1,m05_a,0.8,34,True,24.188,5.5
2,m05_b,1.0,13,True,19.816,1.2
3,m05_b,0.8,42,True,13.379,6.2
4,m05_c,1.0,12,True,21.912,1.1
5,m05_c,0.8,39,True,15.521,5.0
6,m10_a,1.0,19,True,9.268,5.2
7,m10_a,0.8,45,True,3.546,19.5
8,m10_b,1.0,46,True,-12.673,15.5
9,m10_b,0.8,84,True,-15.692,46.9


In [8]:
e = load('m05_a'); r = value_iteration(e, p=0.8)
print(e.render(policy=r.policy))
ep = run_episode(e, r.policy, 0.8, random.Random(42))
print('\nОролдлого:', ep['success'], ep['steps'], 'алхам, шагнал', ep['return'])

S→→↓↓
↓##↓↓
↓##→↓
→→↓#↓
→→→→G

Оролдлого: True 10 алхам, шагнал 41.0


Зураг: `results/astar_paths.png`, `results/vi_convergence.png`, `results/vi_policy_p08.png`.

**Дараагийн (XIV) долоо хоног:** p=1 дээр A* ба VI-г харьцуулах, p=0.8 үед 9×100 = 900 оролдлого → CSV.